# 📅 Dia 2 - Exercício 6: O Seguidor de Cor - Filtro Básico 🎨🔍

Para que o JetRacer consiga seguir uma linha na pista ou reagir a um sinal de trânsito, ele primeiro precisa de saber **isolar essa cor** do resto do cenário (como as paredes da sala, as cadeiras, etc.).

Em Visão por Computador, usamos o formato **HSV** (Matiz, Saturação, Valor) em vez do RGB, porque o HSV ignora sombras e alterações de luz com muito mais facilidade.

---

### 🎯 O Teu Objetivo
Ajustar as barras deslizantes (Sliders) para encontrar os limites exatos da cor de um objeto (ex: uma folha vermelha ou fita amarela). 
* Tudo o que for a cor escolhida deve ficar **BRANCO** no ecrã.
* Tudo o resto deve ficar **PRETO**.

### 🛠️ Instruções Passo a Passo

1. **Coloca um objeto colorido** a cerca de 30cm da câmara do robô.
2. **Executa a célula** abaixo para abrir os dois ecrãs (Imagem Real vs Imagem Filtrada) e os Sliders.
3. **Ajusta os Sliders** de `H_min` (Matiz mínima) até `H_max` (Matiz máxima) devagar para isolar a cor do teu objeto.
   * *Dica para o Vermelho:* Tenta `H_min = 0` e `H_max = 10`.
   * *Dica para o Azul:* Tenta `H_min = 100` e `H_max = 130`.
4. Quando terminares, clica no botão **❌ DESLIGAR FILTRO**.

In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import traitlets
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg

print("--- MÓDULO DE FILTRAGEM DE COR HSV ACTIVADO ---")

# 1. Inicializar a câmara
camera = CSICamera(width=300, height=300, capture_width=1280, capture_height=720, capture_fps=21, flip_method=0)

# 2. Criar os Widgets Visuais
imagem_real_widget = widgets.Image(format='jpeg', width=300, height=300, description="Real")
imagem_filtro_widget = widgets.Image(format='jpeg', width=300, height=300, description="Filtro")

# Sliders para os alunos controlarem o Matiz (Hue) da cor
slider_h_min = widgets.IntSlider(value=0, min=0, max=179, description='H Mínimo:')
slider_h_max = widgets.IntSlider(value=179, min=0, max=179, description='H Máximo:')
botao_desligar = widgets.Button(description="❌ DESLIGAR FILTRO", button_style='danger')

# Mostrar a interface organizada (Ecrãs lado a lado, controlos abaixo)
ecrans = widgets.HBox([imagem_real_widget, imagem_filtro_widget])
controlos = widgets.VBox([slider_h_min, slider_h_max, botao_desligar])
display(ecrans, controlos)

# 3. Link direto da câmara para o ecrã real
link_real = traitlets.dlink((camera, 'value'), (imagem_real_widget, 'value'), transform=bgr8_to_jpeg)

# 4. Função de processamento: pega no frame da câmara, aplica o filtro e atualiza o ecrã preto/branco
def processar_e_filtrar(change):
    frame = change['new']
    
    # Converte para o espaço de cor HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Lê os valores atuais dos sliders dos alunos
    baixo = np.array([slider_h_min.value, 100, 100])
    alto = np.array([slider_h_max.value, 255, 255])
    
    # Cria a máscara binária (Preto e Branco)
    mascara = cv2.inRange(hsv, baixo, alto)
    
    # Envia a máscara convertida em JPEG para o segundo widget
    _, jpeg = cv2.imencode('.jpg', mascara)
    imagem_filtro_widget.value = jpeg.tobytes()

# Diz ao objeto da câmara para chamar a função acima sempre que houver uma imagem nova
camera.observe(processar_e_filtrar, names='value')

# 5. Função de paragem limpa
def desligar_tudo(b):
    camera.unobserve(processar_e_filtrar, names='value')
    link_real.unlink()
    camera.running = False
    botao_desligar.description = "🛑 FILTRO DESLIGADO"
    botao_desligar.button_style = "info"
    botao_desligar.disabled = True
    print("Módulo encerrado com sucesso.")

botao_desligar.on_click(desligar_tudo)

# 6. Ligar o motor de captura da câmara
camera.running = True
print("Sistema pronto! Mexe nos Sliders para isolar uma cor.")